# 大数据智能分析 - Jupyter Notebook

本Notebook演示如何使用PySpark处理大数据集，并与HDFS和MySQL交互。

## 1. 初始化Spark会话

In [ ]:
from pyspark.sql import SparkSession
import os

# 创建Spark会话
spark = SparkSession.builder \
    .appName("BigDataDemo") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .config("spark.sql.warehouse.dir", "hdfs://namenode:9000/user/hive/warehouse") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark会话已创建")

## 2. 从HDFS读取数据

In [ ]:
# 列出HDFS中的数据集
from hdfs import InsecureClient

hdfs_client = InsecureClient('http://namenode:9870')
datasets = hdfs_client.list('/datasets')
print("HDFS中的数据集:")
for dataset in datasets:
    print(f"  - {dataset}")

In [ ]:
# 读取电影元数据集
movie_df = spark.read.csv("hdfs://namenode:9000/datasets/movies_metadata.csv", 
                         header=True, inferSchema=True)
print(f"电影元数据集: {movie_df.count()} 条记录")
movie_df.show(5)

## 3. 数据分析与处理

In [ ]:
from pyspark.sql.functions import col, avg, count, desc

# 计算每部电影的平均评分
movie_stats = movie_df.groupBy("id", "title") \
    .agg(avg("rating").alias("avg_rating"), 
         count("rating").alias("rating_count")) \
    .orderBy(desc("avg_rating"))

movie_stats.show(10)

In [ ]:
# 按电影类型统计
genre_stats = movie_df.groupBy("genre") \
    .agg(avg("rating").alias("avg_rating"), 
         count("rating").alias("rating_count")) \
    .orderBy(desc("avg_rating"))

genre_stats.show()

## 4. 读取电商交易数据

In [ ]:
ecommerce_df = spark.read.csv("hdfs://namenode:9000/datasets/ecommerce_transactions.csv",
                             header=True, inferSchema=True)
print(f"电商交易数据集: {ecommerce_df.count()} 条记录")
ecommerce_df.show(5)

In [ ]:
from pyspark.sql.functions import sum

# 按商品类别统计销售额
category_sales = ecommerce_df.groupBy("category") \
    .agg(sum("total_amount").alias("total_sales"),
         count("transaction_id").alias("transaction_count")) \
    .orderBy(desc("total_sales"))

category_sales.show()

## 5. 连接到MySQL数据库

In [ ]:
# 从MySQL读取聚合结果
mysql_url = "jdbc:mysql://mysql:3306/bigdata_demo"
mysql_properties = {
    "user": "demo",
    "password": "demopassword",
    "driver": "com.mysql.cj.jdbc.Driver"
}

# 读取电影评分聚合表
mysql_movie_agg = spark.read.jdbc(mysql_url, "movies_metadata", properties=mysql_properties)
print("MySQL中的电影评分聚合数据:")
mysql_movie_agg.show(10)

In [ ]:
# 将Spark处理结果保存到MySQL
movie_stats.write \
    .mode("overwrite") \
    .jdbc(mysql_url, "movie_stats_from_notebook", properties=mysql_properties)

print("数据已保存到MySQL")

## 6. 可视化分析

In [ ]:
# 转换为Pandas DataFrame进行可视化
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# 获取电影评分数据
movie_pandas = movie_stats.limit(20).toPandas()

# 创建图表
plt.figure(figsize=(12, 6))
plt.bar(movie_pandas['title'], movie_pandas['avg_rating'])
plt.xlabel('电影名称')
plt.ylabel('平均评分')
plt.title('电影评分TOP 20')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 商品类别销售额饼图
category_pandas = category_sales.toPandas()

plt.figure(figsize=(10, 8))
plt.pie(category_pandas['total_sales'], 
        labels=category_pandas['category'], 
        autopct='%1.1f%%')
plt.title('商品类别销售额分布')
plt.show()

## 7. Spark SQL查询

In [ ]:
# 创建临时视图
movie_df.createOrReplaceTempView("movies_metadata")
ecommerce_df.createOrReplaceTempView("ecommerce")

# 使用Spark SQL查询
top_movies = spark.sql("""
    SELECT title, genre, AVG(rating) as avg_rating, COUNT(*) as rating_count
    FROM movies_metadata
    GROUP BY title, genre
    HAVING COUNT(*) >= 100
    ORDER BY avg_rating DESC
    LIMIT 10
""")

top_movies.show()

## 8. 保存结果到HDFS

In [ ]:
# 将处理结果保存回HDFS
output_path = "hdfs://namenode:9000/results/notebook_output"
movie_stats.write.mode("overwrite").csv(output_path, header=True)
print(f"结果已保存到: {output_path}")

## 9. 清理资源

In [ ]:
# 停止Spark会话
spark.stop()
print("Spark会话已停止")